# OpenInsight Data Ingestion Pipeline - Kaggle Edition

This notebook runs the full OpenInsight ingestion pipeline on Kaggle GPU and writes data to Zilliz Cloud, MongoDB Atlas, and Kaggle working storage.

**Supported sources:** pubmed, medquad, icmr, cochrane, nmc_guideline, rssdi, who, cdc, statpearls

## Where to put API keys

In Kaggle, open the notebook sidebar and add each secret under **Add-ons -> Secrets**. Use the exact secret names below so the notebook can read them automatically:

- `ZILLIZ_URI`
- `ZILLIZ_TOKEN`
- `MONGODB_URL`
- `MONGODB_DB`
- `NVIDIA_NIM_API_KEY`
- `HF_API_TOKEN`  
- `COHERE_API_KEY`  
- `NCBI_API_KEY`  
- `NCBI_EMAIL`

## How to run it

1. Run cells top-to-bottom.
2. For PubMed: upload XMLs as a Kaggle dataset and set `DATA_DIR`.
3. For MedQuAD: just run the MedQuAD download cell, it clones the repo automatically.
4. `EMBED_PROVIDER=local` and `RERANK_PROVIDER=local` use Kaggle GPU.
- Persistent files go under `/kaggle/working`.

## 1. Install Dependencies

In [ ]:
# ── Fix NumPy ABI compatibility (Kaggle ships numpy 2.x, pymilvus needs 1.x) ──
# Must force-reinstall pandas/pymilvus AFTER downgrading numpy so their
# C extensions are rebuilt against the correct numpy headers.
%pip install -q 'numpy<2.0'
%pip install -q --force-reinstall --no-deps pandas

# Install OpenInsight dependencies
%pip install -q \
    fastapi uvicorn python-dotenv pydantic pydantic-settings \
    pymongo motor pypdf2 pdfplumber beautifulsoup4 requests httpx lxml biopython \
    sentence-transformers transformers torch \
    'pymilvus>=2.5,<2.6' langchain langchain-community openai redis \
    tqdm loguru tenacity python-slugify pytesseract Pillow scispacy

# Verify numpy version (must be <2.0)
import numpy as np
print(f'numpy: {np.__version__}')
import pandas as pd
print(f'pandas: {pd.__version__}')
from pymilvus import MilvusClient
print(f'pymilvus: OK')

# Kaggle system packages for GROBID and OCR
!apt-get update -qq
!apt-get install -qq -y default-jre wget unzip tesseract-ocr > /dev/null 2>&1

import os
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/openinsight')
if REPO_DIR.exists():
    %cd /kaggle/working/openinsight
    !git pull origin restruct
else:
    !git clone -b restruct https://github.com/Adi103-ETAI/openinsight.git /kaggle/working/openinsight

sys.path.insert(0, str(REPO_DIR))
print('Environment ready.')


## 2. Configure Kaggle Secrets and Environment

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
from pathlib import Path


def get_secret(name: str, default: str = '') -> str:
    try:
        client = UserSecretsClient()
        value = client.get_secret(name)
        return default if value is None else value
    except Exception:
        return default


# API holders / service secrets
ZILLIZ_URI         = get_secret('ZILLIZ_URI', '')
ZILLIZ_TOKEN       = get_secret('ZILLIZ_TOKEN', '')
MONGODB_URL        = get_secret('MONGODB_URL', '')
MONGODB_DB         = get_secret('MONGODB_DB', 'openinsight')
NVIDIA_NIM_API_KEY = get_secret('NVIDIA_NIM_API_KEY', '')
HF_API_TOKEN       = get_secret('HF_API_TOKEN', '')
COHERE_API_KEY     = get_secret('COHERE_API_KEY', '')
NCBI_API_KEY       = get_secret('NCBI_API_KEY', '')
NCBI_EMAIL         = get_secret('NCBI_EMAIL', 'sentarc.ai@gmail.com')

# ── Environment variables for the pipeline ──
# Kaggle has GPU, so use local providers for ingestion
os.environ['MONGODB_URL']            = MONGODB_URL
os.environ['MONGODB_DB']             = MONGODB_DB
os.environ['VECTOR_URI']             = ZILLIZ_URI
os.environ['VECTOR_TOKEN']           = ZILLIZ_TOKEN
os.environ['MILVUS_CLOUD']           = 'true'
os.environ['VECTOR_COLLECTION_V2']   = 'openinsight_v2'
os.environ['NVIDIA_NIM_API_KEY']     = NVIDIA_NIM_API_KEY
os.environ['NVIDIA_NIM_BASE_URL']    = 'https://integrate.api.nvidia.com/v1'
os.environ['NIM_MODEL']              = 'meta/llama-3.1-70b-instruct'
os.environ['EMBED_PROVIDER']         = 'local'   # Kaggle has GPU -> use SentenceTransformers locally
os.environ['RERANK_PROVIDER']        = 'local'   # Kaggle has GPU -> use CrossEncoder locally
os.environ['HF_API_TOKEN']           = HF_API_TOKEN
os.environ['DENSE_MODEL_NAME']       = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['HF_EMBED_MODEL']         = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['RERANKER_MODEL_NAME']    = 'BAAI/bge-reranker-v2-m3'
os.environ['COHERE_API_KEY']         = COHERE_API_KEY
os.environ['NCBI_API_KEY']           = NCBI_API_KEY
os.environ['NCBI_EMAIL']             = NCBI_EMAIL
os.environ['SSL_CERT_FILE']          = __import__('certifi').where()

# Directories
PUBMED_DATA_DIR = '/kaggle/input/your-dataset-name'  # UPDATE for PubMed XMLs
MEDQUAD_DATA_DIR = '/kaggle/working/medquad_data'

print('Configuration loaded.')
print(f'Embed provider: {os.environ["EMBED_PROVIDER"]}')
print(f'Rerank provider: {os.environ["RERANK_PROVIDER"]}')
print(f'Zilliz URI set: {bool(ZILLIZ_URI)}')
print(f'MongoDB URL set: {bool(MONGODB_URL)}')

## 3. Start GROBID and Verify GPU

In [ ]:
import subprocess
import time
import httpx
import torch

GROBID_DIR = Path('/kaggle/working/grobid')
GROBID_BIN = GROBID_DIR / 'grobid-0.8.1' / 'bin' / 'grobid-core'
GROBID_URL = 'http://localhost:8070'

if not GROBID_BIN.exists():
    !wget -q https://github.com/kermitt2/grobid/releases/download/0.8.1/grobid-0.8.1.zip -O /kaggle/working/grobid-0.8.1.zip
    !unzip -q -o /kaggle/working/grobid-0.8.1.zip -d /kaggle/working/grobid

if 'grobid_proc' not in globals() or grobid_proc.poll() is not None:
    grobid_proc = subprocess.Popen(
        [str(GROBID_BIN), '--port', '8070'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

os.environ['GROBID_URL'] = GROBID_URL

for _ in range(30):
    try:
        response = httpx.get(f'{GROBID_URL}/api/isalive', timeout=5)
        if response.status_code == 200:
            print('GROBID status: alive')
            break
    except Exception:
        time.sleep(2)
else:
    print('GROBID did not respond after waiting.')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4.5. Wipe Old Data (First Run Only)

Run this cell **once** before your first ingestion to clear any old/bad data.

In [ ]:
# ── WIPE ALL DATA (Run this once before first ingestion) ──
from pymilvus import MilvusClient
from pymongo import MongoClient

zilliz = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz.has_collection('openinsight_v2'):
    zilliz.drop_collection('openinsight_v2')
    print('Zilliz: Dropped collection openinsight_v2')
else:
    print('Zilliz: Collection does not exist yet')

mongo = MongoClient(MONGODB_URL)
db = mongo[MONGODB_DB]
for col_name in ['documents_v2', 'chunks_v2', 'failed_documents',
                  'ingestion_checkpoints', 'ingestion_metrics']:
    result = db[col_name].delete_many({})
    print(f'MongoDB {col_name}: Deleted {result.deleted_count} docs')

print('\nAll data wiped! Ready for fresh ingestion.')

## 4a. Ingest PubMed Data

Set `PUBMED_DATA_DIR` above to your Kaggle dataset path containing PubMed/StatPearls XML files.

In [ ]:
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()

summary = await pipeline.ingest_directory(
    directory=PUBMED_DATA_DIR,
    source='pubmed',
    recreate_index=True,
    batch_size=10,
    resume=True,
)

print('\nINGESTION COMPLETE (PubMed)')
for key, value in summary.items():
    print(f'  {key}: {value}')

## 4b. Download & Ingest MedQuAD Data

Clones the MedQuAD repo (47K medical QA pairs from 12 NIH sources) and ingests the non-empty collections.
The parser automatically skips QA pairs with empty answers (copyright-stripped collections).

In [ ]:
# ── Download MedQuAD ──
import os
from pathlib import Path

MEDQUAD_DIR = Path(MEDQUAD_DATA_DIR)
MEDQUAD_REPO = MEDQUAD_DIR / 'MedQuAD'

if MEDQUAD_REPO.exists():
    print(f'MedQuAD already cloned at {MEDQUAD_REPO}')
else:
    MEDQUAD_DIR.mkdir(parents=True, exist_ok=True)
    !git clone https://github.com/abachaa/MedQuAD.git {MEDQUAD_REPO}
    print('MedQuAD cloned.')

# Show available collections and file counts
xml_files = list(MEDQUAD_REPO.rglob('*.xml'))
print(f'Total XML files: {len(xml_files)}')
for subdir in sorted(MEDQUAD_REPO.iterdir()):
    if subdir.is_dir():
        count = len(list(subdir.rglob('*.xml')))
        print(f'  {subdir.name}: {count} XML files')

In [ ]:
# ── Ingest MedQuAD ──
# The pipeline uses MedQuADParser which auto-skips empty-answer QA pairs.
# Only these collections have full content:
#   2_GARD_QA, 3_GHR_QA, 4_NIDDK_QA, 5_NINDS_QA, 6_NHLBI_QA,
#   7_CDC_QA, 8_SeniorHealth_QA, 9_MPlusHealthTopics_QA, 10_CancerGov_QA
# Collections 1_ADA, 11_MPlusDrugs, 12_MPlusHerbSupp have empty answers.

from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()

# Ingest all MedQuAD XML files (parser filters empty answers automatically)
summary = await pipeline.ingest_directory(
    directory=str(MEDQUAD_REPO),
    source='medquad',
    recreate_index=False,  # Don't wipe — append to existing collection
    batch_size=50,
    resume=True,
)

print('\nINGESTION COMPLETE (MedQuAD)')
for key, value in summary.items():
    print(f'  {key}: {value}')

## 5. Verify Storage and Metrics

In [ ]:
from pymilvus import MilvusClient
from pymongo import MongoClient

collection_name = os.environ.get('VECTOR_COLLECTION_V2', 'openinsight_v2')

zilliz_client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz_client.has_collection(collection_name):
    stats = zilliz_client.get_collection_stats(collection_name)
    print(f'Zilliz collection: {collection_name}')
    print(f"Row count: {stats.get('row_count', 'unknown')}")
else:
    print(f'Collection not found: {collection_name}')

mongo_client = MongoClient(MONGODB_URL)
db = mongo_client[MONGODB_DB]
print(f'\nMongoDB database: {MONGODB_DB}')
print(f"documents_v2: {db.documents_v2.count_documents({})}")
print(f"chunks_v2: {db.chunks_v2.count_documents({})}")
print(f"failed_documents: {db.failed_documents.count_documents({})}")

# Show per-source breakdown
pipeline_names = db.documents_v2.distinct('source_type')
print(f'\nSources in DB: {pipeline_names}')
for src in pipeline_names:
    count = db.documents_v2.count_documents({'source_type': src})
    print(f'  {src}: {count} documents')

# Optional cleanup
try:
    grobid_proc.terminate()
    print('\nGROBID stopped.')
except Exception:
    pass